# 03 · Train the proposed model (full `PENet`)
Trains the vessel-topology-aware multi-task network with all components enabled, selects decision thresholds on the **validation** set, and saves the checkpoint + thresholds used by notebooks 04–07.

Typical Kaggle settings: GPU T4/P100, `cfg.vol_shape=(64,128,128)`, `bs=2`, `accum=4`. Training time scales with the number of studies; do a smoke test with `SUBSET=64` first.

In [ ]:
%run ./00_common.ipynb

In [ ]:
sp = load_splits()
SUBSET = None                                   # e.g. 64 for a smoke test
tr, va, te = sp["train"][:SUBSET], sp["val"][:SUBSET], sp["test"][:SUBSET]
fl = Flags(); model = PENet(cfg, fl).to(DEVICE)
print(f"{fl.name}\nparams: {count_params(model):.2f} M")

# --- shape / loss sanity check on one batch before the long run
b = to_dev(next(iter(make_loader(tr[:cfg.bs * 2], cfg, True))))
out = model(b); loss, parts = compute_loss(out, b, cfg)
print({k: tuple(v.shape) for k, v in out.items() if torch.is_tensor(v)}); print("loss", float(loss), parts)

In [ ]:
set_seed(cfg.seed)
model, hist, best = train_model(model, tr, va, cfg, epochs=cfg.epochs, tag="penet_full")
import pandas as pd; H = pd.DataFrame(hist); H.to_csv(f"{cfg.out_dir}/penet_full_history.csv", index=False); print("best val AUC:", best)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for c in [c for c in H.columns if c.startswith("loss_")]: ax[0].plot(H[c], label=c[5:])
ax[0].set_title("training loss terms"); ax[0].legend(fontsize=7)
ax[1].plot(H.val_auc, label="val AUC"); ax[1].plot(H.val_ap, label="val AP"); ax[1].legend(); ax[1].set_title("detection")
ax[2].plot(H.val_slice_f1); ax[2].set_title("val slice-level F1 (localisation)"); plt.tight_layout(); plt.show()

### Threshold selection on validation set → saved for all later notebooks

In [ ]:
Rv = predict(model, make_loader(va, cfg, False))
thr_slice, f1 = best_slice_thr(Rv["slice_prob"], Rv["slice_lab"], Rv["slice_valid"])
from sklearn.metrics import roc_curve
fpr, tpr, th = roc_curve(Rv["y"][:, 0], Rv["det_prob"]); thr_det = float(th[np.argmax(tpr - fpr)])      # Youden J
json.dump(dict(thr_det=thr_det, thr_slice=thr_slice), open(f"{cfg.ckpt_dir}/thresholds.json", "w"))
print(f"thr_det={thr_det:.3f}  thr_slice={thr_slice:.2f} (val slice F1={f1:.3f})")
Rt = predict(model, make_loader(te, cfg, False)); print("TEST", det_metrics(Rt))

In [1]:
from pathlib import Path
from shutil import disk_usage
import subprocess

import psutil
import tensorflow as tf
from google.colab import drive

drive.mount("/content/drive")

DATASET_DIR = Path("/content/drive/MyDrive/Pulmonary_Embolism_Dataset")
assert DATASET_DIR.is_dir(), f"Dataset folder not found: {DATASET_DIR}"

expected_names = {f"train-{i:03d}-1.tfrec" for i in range(30)}
actual_names = {p.name for p in DATASET_DIR.glob("*.tfrec") if p.is_file()}
assert actual_names == expected_names, (
    f"Missing: {sorted(expected_names - actual_names)}\n"
    f"Unexpected: {sorted(actual_names - expected_names)}"
)

GIB = 1024**3
ram = psutil.virtual_memory()
disk = disk_usage("/content")

print(f"Verified shards: {len(actual_names)}")
print(f"TensorFlow: {tf.__version__}")
print(f"RAM available: {ram.available / GIB:.1f} / {ram.total / GIB:.1f} GiB")
print(f"Scratch disk free: {disk.free / GIB:.1f} / {disk.total / GIB:.1f} GiB")
print("TensorFlow GPUs:", tf.config.list_physical_devices("GPU"))

try:
    gpu = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,memory.free",
            "--format=csv",
        ],
        capture_output=True,
        text=True,
        check=False,
    )
    print(gpu.stdout.strip() if gpu.returncode == 0 else gpu.stderr.strip())
except FileNotFoundError:
    print("nvidia-smi is unavailable. The data check can still run on CPU.")

Mounted at /content/drive
Verified shards: 30
TensorFlow: 2.21.0
RAM available: 11.0 / 12.7 GiB
Scratch disk free: 112.5 / 112.6 GiB
TensorFlow GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
name, memory.total [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 14910 MiB


In [2]:
FEATURES = {
    key: tf.io.FixedLenFeature([], tf.string)
    for key in ("image", "image_name", "target")
}
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 8

def parse_record(serialized):
    record = tf.io.parse_single_example(serialized, FEATURES)
    target = tf.io.parse_tensor(record["target"], out_type=tf.uint8)
    target = tf.ensure_shape(target, (10,))
    label = tf.cast(target[0], tf.int32)

    tf.debugging.assert_less_equal(label, 1, message="target[0] must be 0 or 1")
    tf.debugging.assert_positive(
        tf.strings.length(record["image_name"]),
        message="image_name is empty",
    )

    rgb = tf.io.decode_jpeg(record["image"], channels=3)
    return rgb, label

def preprocess_grayscale(rgb, label):
    image = tf.image.rgb_to_grayscale(tf.image.convert_image_dtype(rgb, tf.float32))
    image = tf.image.resize(image, IMAGE_SIZE, antialias=True)
    return tf.ensure_shape(image, (*IMAGE_SIZE, 1)), label

In [3]:
from collections import Counter

import numpy as np

SAMPLE_SHARDS = [0, 9, 19, 29]
RECORDS_PER_SHARD = 32

class_counts = Counter()
raw_shapes = Counter()
valid_rgb = []
valid_labels = []
channel_spreads = []
record_errors = 0
shard_errors = 0

for shard_index in SAMPLE_SHARDS:
    path = DATASET_DIR / f"train-{shard_index:03d}-1.tfrec"
    shard_counts = Counter()
    sample = tf.data.TFRecordDataset(str(path)).take(RECORDS_PER_SHARD)

    try:
        for offset, raw in enumerate(sample):
            try:
                rgb, label = parse_record(raw)
            except (tf.errors.OpError, ValueError) as exc:
                record_errors += 1
                if record_errors <= 5:
                    print(f"{path.name}, record {offset}: {str(exc).splitlines()[0]}")
                continue

            label_value = int(label.numpy())
            shard_counts[label_value] += 1
            raw_shapes[tuple(rgb.shape)] += 1

            pixels = rgb.numpy().astype(np.int16)
            spread = pixels.max(axis=-1) - pixels.min(axis=-1)
            channel_spreads.append(float(spread.mean()))

            if len(valid_rgb) < BATCH_SIZE:
                valid_rgb.append(rgb)
                valid_labels.append(label)

    except tf.errors.OpError as exc:
        shard_errors += 1
        print(f"{path.name}: shard read error: {str(exc).splitlines()[0]}")

    class_counts.update(shard_counts)
    print(f"{path.name}: valid={sum(shard_counts.values())}, labels={dict(shard_counts)}")

print("\nSample label counts:", {label: class_counts[label] for label in (0, 1)})
print("Decoded image shapes:", dict(raw_shapes))
print(f"Record errors: {record_errors} | Shard read errors: {shard_errors}")

if channel_spreads:
    print(
        "Per-image mean RGB channel spread (min / median / max, 0–255 scale):",
        np.round(np.percentile(channel_spreads, [0, 50, 100]), 3),
    )

train-000-1.tfrec: valid=32, labels={0: 32}
train-009-1.tfrec: valid=32, labels={0: 32}
train-019-1.tfrec: valid=32, labels={0: 19, 1: 13}
train-029-1.tfrec: valid=32, labels={0: 29, 1: 3}

Sample label counts: {0: 112, 1: 16}
Decoded image shapes: {(256, 256, 3): 128}
Record errors: 0 | Shard read errors: 0
Per-image mean RGB channel spread (min / median / max, 0–255 scale): [ 69.598  80.708 101.841]


In [4]:
assert len(valid_rgb) == BATCH_SIZE, "Fewer than 8 valid examples. Check Cell 3."

examples = [
    preprocess_grayscale(rgb, label)
    for rgb, label in zip(valid_rgb, valid_labels)
]
images = tf.stack([image for image, _ in examples])
labels = tf.stack([label for _, label in examples])

assert tuple(images.shape) == (8, 256, 256, 1)
assert images.dtype == tf.float32
assert tuple(labels.shape) == (8,)
assert labels.dtype == tf.int32

tf.debugging.assert_all_finite(images, "Batch contains non-finite pixels")
tf.debugging.assert_greater_equal(images, 0.0)
tf.debugging.assert_less_equal(images, 1.0 + 1e-6)
tf.debugging.assert_greater_equal(labels, 0)
tf.debugging.assert_less_equal(labels, 1)

print("Image batch:", tuple(images.shape), images.dtype.name)
print("Pixel range:", float(tf.reduce_min(images)), float(tf.reduce_max(images)))
print("Label batch:", tuple(labels.shape), labels.numpy().tolist())
print("Batch check passed.")

Image batch: (8, 256, 256, 1) float32
Pixel range: 0.0 0.9998999834060669
Label batch: (8,) [0, 0, 0, 0, 0, 0, 0, 0]
Batch check passed.


In [5]:
from collections import Counter
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
import json
import random
import shutil
import sqlite3
import time
import uuid

SEED = 42
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
LOCAL_DIR = Path("/content/pe_inventory") / RUN_ID
DRIVE_DIR = Path("/content/drive/MyDrive/Pulmonary_Embolism_Research/splits") / RUN_ID

LOCAL_DIR.mkdir(parents=True, exist_ok=False)
DB_PATH = LOCAL_DIR / "split_manifest.sqlite"
SHARDS = sorted(DATASET_DIR.glob("train-*.tfrec"))

expected = {f"train-{i:03d}-1.tfrec" for i in range(30)}
assert {path.name for path in SHARDS} == expected, "Expected exactly the 30 training shards."

print("Run:", RUN_ID)
print("Shards:", len(SHARDS))
print("Drive output after successful verification:", DRIVE_DIR)

Run: 20261118T174449Z-274e4b1b
Shards: 30
Drive output after successful verification: /content/drive/MyDrive/Pulmonary_Embolism_Research/splits/20261118T174449Z-274e4b1b


In [ ]:
if DB_PATH.exists():
    raise FileExistsError("Run Cell 5 again before starting a fresh inventory.")

scan_started = time.monotonic()
db = sqlite3.connect(DB_PATH)

db.executescript("""
CREATE TABLE records (
    id INTEGER PRIMARY KEY,
    shard TEXT NOT NULL,
    record_index INTEGER NOT NULL,
    image_name BLOB NOT NULL,
    label INTEGER NOT NULL CHECK (label IN (0, 1)),
    pixel_hash BLOB NOT NULL,
    UNIQUE (shard, record_index)
);
CREATE TABLE invalid_records (
    shard TEXT NOT NULL,
    record_index INTEGER NOT NULL,
    error_type TEXT NOT NULL
);
CREATE TABLE shard_stats (
    shard TEXT PRIMARY KEY,
    size_bytes INTEGER NOT NULL,
    valid_negative INTEGER NOT NULL,
    valid_positive INTEGER NOT NULL,
    invalid INTEGER NOT NULL
);
""")

def flush_inventory_rows(db, valid_rows, invalid_rows):
    db.executemany(
        """INSERT INTO records
           (shard, record_index, image_name, label, pixel_hash)
           VALUES (?, ?, ?, ?, ?)""",
        valid_rows,
    )
    db.executemany("INSERT INTO invalid_records VALUES (?, ?, ?)", invalid_rows)
    valid_rows.clear()
    invalid_rows.clear()

try:
    for shard_number, path in enumerate(SHARDS, start=1):
        counts = Counter()
        valid_rows, invalid_rows = [], []

        try:
            with db:
                records = tf.data.TFRecordDataset(str(path))
                for offset, raw in enumerate(records):
                    try:
                        rgb, label = parse_record(raw)
                        fields = tf.io.parse_single_example(raw, FEATURES)
                        name = fields["image_name"].numpy()

                        pixels = rgb.numpy()
                        digest = sha256()
                        digest.update(f"{pixels.shape}\n".encode("ascii"))
                        digest.update(pixels.tobytes())
                        label_value = int(label.numpy())

                        valid_rows.append(
                            (path.name, offset, name, label_value, digest.digest())
                        )
                        counts[label_value] += 1
                    except (tf.errors.InvalidArgumentError, tf.errors.DataLossError, ValueError) as exc:
                        invalid_rows.append((path.name, offset, type(exc).__name__))
                        counts["invalid"] += 1

                    if len(valid_rows) + len(invalid_rows) >= 4096:
                        flush_inventory_rows(db, valid_rows, invalid_rows)

                flush_inventory_rows(db, valid_rows, invalid_rows)
                db.execute(
                    "INSERT INTO shard_stats VALUES (?, ?, ?, ?, ?)",
                    (path.name, path.stat().st_size, counts[0], counts[1], counts["invalid"]),
                )
        except tf.errors.OpError as exc:
            raise RuntimeError(f"Could not finish reading {path.name}. No split published.") from exc

        elapsed_minutes = (time.monotonic() - scan_started) / 60
        print(
            f"[{shard_number:02d}/30] {path.name} | "
            f"negative={counts[0]:,} positive={counts[1]:,} "
            f"invalid={counts['invalid']:,} | elapsed={elapsed_minutes:.1f} min",
            flush=True,
        )
finally:
    db.close()

print("Inventory complete. Run Cell 7 to build and verify the split.")

[01/30] train-000-1.tfrec | negative=56,674 positive=3,012 invalid=0 | elapsed=6.3 min
[02/30] train-001-1.tfrec | negative=56,623 positive=3,063 invalid=0 | elapsed=12.4 min
[03/30] train-002-1.tfrec | negative=56,856 positive=2,830 invalid=0 | elapsed=18.5 min
[04/30] train-003-1.tfrec | negative=56,417 positive=3,269 invalid=0 | elapsed=24.6 min
[05/30] train-004-1.tfrec | negative=56,473 positive=3,213 invalid=0 | elapsed=30.7 min


In [ ]:
from array import array

with sqlite3.connect(DB_PATH) as db:
    completed_shards = db.execute("SELECT COUNT(*) FROM shard_stats").fetchone()[0]
    if completed_shards != len(SHARDS):
        raise RuntimeError(f"Inventory incomplete: {completed_shards}/{len(SHARDS)} shards.")

    max_id = db.execute("SELECT COALESCE(MAX(id), 0) FROM records").fetchone()[0]
    parent = array("q", range(max_id + 1))
    rank = bytearray(max_id + 1)

    def find_group(record_id):
        while parent[record_id] != record_id:
            parent[record_id] = parent[parent[record_id]]
            record_id = parent[record_id]
        return record_id

    def merge_groups(first, second):
        a, b = find_group(first), find_group(second)
        if a == b:
            return
        if rank[a] < rank[b]:
            a, b = b, a
        parent[b] = a
        if rank[a] == rank[b]:
            rank[a] += 1

    db.execute("CREATE INDEX IF NOT EXISTS records_by_name ON records(image_name, id)")
    db.execute("CREATE INDEX IF NOT EXISTS records_by_pixels ON records(pixel_hash, id)")

    # Adjacent rows with the same sorted key link every member of that group.
    for column in ("image_name", "pixel_hash"):
        previous_key = None
        previous_id = None
        for record_id, key in db.execute(
            f"SELECT id, {column} FROM records ORDER BY {column}, id"
        ):
            if key == previous_key:
                merge_groups(record_id, previous_id)
            previous_key, previous_id = key, record_id

    db.execute("DROP TABLE IF EXISTS membership")
    db.execute("CREATE TABLE membership (id INTEGER PRIMARY KEY, group_id INTEGER NOT NULL)")
    db.executemany(
        "INSERT INTO membership VALUES (?, ?)",
        ((record_id, find_group(record_id))
         for (record_id,) in db.execute("SELECT id FROM records ORDER BY id")),
    )

    db.execute("DROP TABLE IF EXISTS image_groups")
    db.execute("""
        CREATE TABLE image_groups AS
        SELECT m.group_id,
               MIN(r.id) AS representative_id,
               MIN(r.label) AS low_label,
               MAX(r.label) AS high_label,
               COUNT(*) AS group_size
        FROM records AS r
        JOIN membership AS m ON m.id = r.id
        GROUP BY m.group_id
    """)

    db.execute("DROP TABLE IF EXISTS splits")
    db.execute("""
        CREATE TABLE splits (
            record_id INTEGER PRIMARY KEY REFERENCES records(id),
            split TEXT NOT NULL CHECK (split IN ('train', 'validation', 'test'))
        )
    """)

    rng = random.Random(SEED)

    for label in (0, 1):
        candidates = [
            record_id for (record_id,) in db.execute("""
                SELECT representative_id FROM image_groups
                WHERE low_label = ? AND high_label = ?
                ORDER BY representative_id
            """, (label, label))
        ]
        rng.shuffle(candidates)

        n = len(candidates)
        n_val = round(n * 0.15)
        n_test = round(n * 0.15)
        n_train = n - n_val - n_test

        if min(n_train, n_val, n_test) < 1:
            raise RuntimeError(f"Too few usable images for label {label}: {n}")

        db.executemany(
            "INSERT INTO splits VALUES (?, ?)",
            (
                (record_id, "train" if i < n_train else
                 "validation" if i < n_train + n_val else "test")
                for i, record_id in enumerate(candidates)
            ),
        )

    raw_counts = dict(db.execute("SELECT label, COUNT(*) FROM records GROUP BY label"))
    invalid_count = db.execute("SELECT COUNT(*) FROM invalid_records").fetchone()[0]
    conflict_count, duplicate_count = db.execute("""
        SELECT
            COALESCE(SUM(CASE WHEN low_label != high_label THEN group_size ELSE 0 END), 0),
            COALESCE(SUM(CASE WHEN low_label = high_label THEN group_size - 1 ELSE 0 END), 0)
        FROM image_groups
    """).fetchone()
    assignments = db.execute("""
        SELECT s.split, r.label, COUNT(*)
        FROM splits AS s JOIN records AS r ON r.id = s.record_id
        GROUP BY s.split, r.label
    """).fetchall()

    split_counts = {split: {} for split in ("train", "validation", "test")}
    for split, label, count in assignments:
        split_counts[split][label] = count

    selected = sum(sum(counts.values()) for counts in split_counts.values())
    assert all(set(counts) == {0, 1} for counts in split_counts.values())
    assert sum(raw_counts.values()) == selected + conflict_count + duplicate_count

    invalid_selection = db.execute("""
        SELECT COUNT(*)
        FROM splits AS s
        JOIN membership AS m ON m.id = s.record_id
        JOIN image_groups AS g ON g.group_id = m.group_id
        WHERE s.record_id != g.representative_id OR g.low_label != g.high_label
    """).fetchone()[0]
    assert invalid_selection == 0, "A selected record violates the duplicate policy."

    assert db.execute("PRAGMA quick_check").fetchone()[0] == "ok"

db.close()

summary = {
    "run_id": RUN_ID,
    "seed": SEED,
    "tensorflow_version": tf.__version__,
    "dataset_path": str(DATASET_DIR),
    "shards": [path.name for path in SHARDS],
    "label": "target[0] / pe_present_on_image",
    "baseline_input": {"size": [256, 256], "channels": 3, "pixel_range": [0, 1]},
    "valid_counts_before_duplicate_handling": raw_counts,
    "invalid_records": invalid_count,
    "conflicting_label_records_excluded": conflict_count,
    "redundant_duplicate_records_excluded": duplicate_count,
    "selected_image_counts": split_counts,
    "split_unit": "image; patient and exam independence unverified",
}

summary_path = LOCAL_DIR / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")

DRIVE_DIR.mkdir(parents=True, exist_ok=False)
drive_manifest = DRIVE_DIR / DB_PATH.name
shutil.copy2(DB_PATH, drive_manifest)

with sqlite3.connect(f"{drive_manifest.as_uri()}?mode=ro", uri=True) as saved_db:
    assert saved_db.execute("SELECT COUNT(*) FROM splits").fetchone()[0] == selected
    assert saved_db.execute("PRAGMA quick_check").fetchone()[0] == "ok"

# The summary appears only after the Drive database passes readback.
shutil.copy2(summary_path, DRIVE_DIR / summary_path.name)
print(json.dumps(summary, indent=2))
print("\nVerified split manifest saved to:", DRIVE_DIR)